In [1]:
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
ruta = Path("/content/drive/MyDrive/TRABAJO2 SI/datamart/mart_comuna_anio.csv")
mart = pd.read_csv(ruta, encoding="utf-8-sig")

print(mart.shape)
print(mart.columns.tolist())

Mounted at /content/drive
(200, 10)
['id_comuna', 'id_tiempo', 'anio', 'n_empresas_agro', 'trab_dependientes_agro', 'trab_ponderados_agro', 'n_incendios', 'sup_afectada_total', 'pct_superficie_cerezo_2024', 'pct_superficie_goteo_2024']


## ML: segmentación jerárquica de comunas

Agruparemos las 10 comunas según su dependencia del empleo agropecuario, exposición histórica a incendios y concentración de cerezo. El porcentaje de riego por goteo se mostrará como indicador de adaptación. Compararemos agrupaciones de 2 a 4 grupos mediante silhouette y revisaremos los perfiles resultantes antes de interpretarlos.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

RUTA_DATAMART = Path("/content/drive/MyDrive/TRABAJO2 SI/datamart")
mart = pd.read_csv(RUTA_DATAMART / "mart_comuna_anio.csv", encoding="utf-8-sig")

# Dependencia agropecuaria en 2024: empleo 011–015 / empleo comunal reportado
sii = (
    pd.read_csv(RUTA_DATAMART / "fact_economia.csv", encoding="utf-8-sig")
    .merge(
        pd.read_csv(RUTA_DATAMART / "dim_actividad.csv", encoding="utf-8-sig")
        [["id_actividad", "codigo_actividad"]],
        on="id_actividad",
        validate="many_to_one"
    )
    .merge(
        pd.read_csv(RUTA_DATAMART / "dim_tiempo.csv", encoding="utf-8-sig"),
        on="id_tiempo",
        validate="many_to_one"
    )
)

sii["codigo6"] = (
    pd.to_numeric(sii["codigo_actividad"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(6)
)
sii["subgrupo"] = sii["codigo6"].str[:3]
sii = sii[
    (sii["anio"] == 2024)
    & sii["subrubro"].str.strip().ne("Sin información")
]

comunas = sorted(mart["id_comuna"].unique())
agro = (
    sii[sii["subgrupo"].between("011", "015").fillna(False)]
    .groupby("id_comuna")["trab_ponderados"].sum()
    .reindex(comunas, fill_value=0)
)
total = sii.groupby("id_comuna")["trab_ponderados"].sum().reindex(comunas)
peso_agro = (agro / total).rename("peso_agro")

# Incendios: promedios anuales 2005–2024
incendios = (
    mart[mart["anio"].between(2005, 2024)]
    .groupby("id_comuna")
    .agg(
        incendios_prom=("n_incendios", "mean"),
        superficie_afectada_prom=("sup_afectada_total", "mean")
    )
)

# Catastro 2024
catastro = mart.groupby("id_comuna").agg(
    cerezo=("pct_superficie_cerezo_2024", "first"),
    goteo=("pct_superficie_goteo_2024", "first")
)

features = (
    pd.DataFrame(index=pd.Index(comunas, name="id_comuna"))
    .join(peso_agro)
    .join(incendios)
    .join(catastro)
    .reset_index()
)

variables = [
    "peso_agro", "incendios_prom",
    "superficie_afectada_prom", "cerezo"
]
X = SimpleImputer(strategy="median").fit_transform(features[variables])
X = StandardScaler().fit_transform(X)

evaluacion = []
for k in (2, 3, 4):
    etiquetas = AgglomerativeClustering(
        n_clusters=k, linkage="ward"
    ).fit_predict(X)
    evaluacion.append((k, silhouette_score(X, etiquetas), etiquetas))

k_final, sil_final, etiquetas = max(evaluacion, key=lambda x: x[1])
features["grupo"] = etiquetas

print("Silhouette por grupos:", [(k, round(s, 3)) for k, s, _ in evaluacion])
print("Grupos elegidos:", k_final, "| Silhouette:", round(sil_final, 3))
display(features.sort_values("grupo"))
display(features.groupby("grupo")[variables + ["goteo"]].mean().round(3))

Silhouette por grupos: [(2, np.float64(0.327)), (3, np.float64(0.335)), (4, np.float64(0.205))]
Grupos elegidos: 3 | Silhouette: 0.335


,id_comuna,peso_agro,incendios_prom,superficie_afectada_prom,cerezo,goteo,grupo
0,1,0.357862,2.894737,88.551000,31.270597,92.178861,0
1,2,0.358200,5.750000,182.376250,58.897338,67.915973,0
2,3,0.339289,6.200000,665.316560,4.835558,99.367960,0
3,4,0.325883,2.333333,12.469500,20.603854,79.556380,0
4,5,0.189681,1.933333,26.813333,33.993591,94.181707,0
5,6,0.151045,4.400000,221.753500,20.007841,89.498110,0
6,7,0.270697,1.500000,4.297895,32.796993,66.547152,0
8,9,0.128317,19.300000,286.800780,57.603961,82.104778,1
9,10,0.063675,9.000000,956.112500,27.202350,91.123669,1
7,8,0.209103,4.850000,2739.801000,7.263167,100.000000,2


,peso_agro,incendios_prom,superficie_afectada_prom,cerezo,goteo
grupo,,,,,
0,0.285,3.573,171.654,28.915,84.178
1,0.096,14.150,621.457,42.403,86.614
2,0.209,4.850,2739.801,7.263,100.000


### Revisar grupos sin que los incendios extremos dominen

La superficie afectada tiene valores muy dispares. Aplicaremos log(1 + x) a las variables de incendios y compararemos agrupaciones de 2 y 3 grupos, junto con sus comunas y silhouette.

In [6]:
X_revision = features[variables].copy()
X_revision[["incendios_prom", "superficie_afectada_prom"]] = np.log1p(
    X_revision[["incendios_prom", "superficie_afectada_prom"]]
)

X_revision = SimpleImputer(strategy="median").fit_transform(X_revision)
X_revision = StandardScaler().fit_transform(X_revision)

nombres = pd.read_csv(
    RUTA_DATAMART / "dim_comuna.csv", encoding="utf-8-sig"
).set_index("id_comuna")["comuna"]

for k in (2, 3):
    grupos = AgglomerativeClustering(
        n_clusters=k, linkage="ward"
    ).fit_predict(X_revision)

    resultado = features[["id_comuna"]].copy()
    resultado["comuna"] = resultado["id_comuna"].map(nombres)
    resultado["grupo"] = grupos

    print(f"k={k} | silhouette={silhouette_score(X_revision, grupos):.3f}")
    display(resultado.sort_values(["grupo", "comuna"]))

k=2 | silhouette=0.264


,id_comuna,comuna,grupo
1,2,Chimbarongo,0
2,3,Lolol,0
5,6,Peralillo,0
7,8,Pumanque,0
8,9,San Fernando,0
9,10,Santa Cruz,0
0,1,Chépica,1
3,4,Nancagua,1
4,5,Palmilla,1
6,7,Placilla,1


k=3 | silhouette=0.314


,id_comuna,comuna,grupo
1,2,Chimbarongo,0
8,9,San Fernando,0
0,1,Chépica,1
3,4,Nancagua,1
4,5,Palmilla,1
6,7,Placilla,1
2,3,Lolol,2
5,6,Peralillo,2
7,8,Pumanque,2
9,10,Santa Cruz,2


### Comparar los perfiles de los grupos

Revisaremos las comunas y los promedios originales de cada grupo para comprobar si la segmentación se puede interpretar.

In [7]:
base_perfiles = resultado[["id_comuna", "comuna", "grupo"]].merge(
    features.drop(columns="grupo", errors="ignore"),
    on="id_comuna",
    how="left"
)

perfiles = base_perfiles.groupby("grupo").agg(
    comunas=("comuna", lambda x: ", ".join(x)),
    n_comunas=("id_comuna", "nunique"),
    peso_agro=("peso_agro", "mean"),
    incendios_prom=("incendios_prom", "mean"),
    superficie_afectada_prom=("superficie_afectada_prom", "mean"),
    cerezo=("cerezo", "mean"),
    goteo=("goteo", "mean")
).round(2)

display(perfiles)

,comunas,n_comunas,peso_agro,incendios_prom,superficie_afectada_prom,cerezo,goteo
grupo,,,,,,,
0,"Chimbarongo, San Fernando",2,0.24,12.52,234.59,58.25,75.01
1,"Chépica, Nancagua, Palmilla, Placilla",4,0.29,2.17,33.03,29.67,83.12
2,"Lolol, Peralillo, Pumanque, Santa Cruz",4,0.19,6.11,1145.75,14.83,95.00


### Interpretación de la segmentación

La opción de 3 grupos obtuvo silhouette 0,314 y formó grupos de 2, 4 y 4 comunas. Los perfiles difieren en dependencia agropecuaria, concentración de cerezo y exposición a incendios:

- Grupo 0: más cerezo e incendios registrados.
- Grupo 1: mayor peso del empleo agro y menor exposición histórica a incendios.
- Grupo 2: mayor superficie afectada por incendios, con menor peso agro y menor concentración de cerezo.

La segmentación sirve para comparar perfiles comunales; no establece un ranking general de riesgo.

In [9]:
salida = RUTA_DATAMART.parent / "ml_segmentacion"
salida.mkdir(parents=True, exist_ok=True)

base_perfiles.to_csv(
    salida / "grupos_comunas.csv",
    index=False,
    encoding="utf-8-sig"
)
perfiles.to_csv(
    salida / "perfiles_grupo.csv",
    encoding="utf-8-sig"
)

print("Archivos guardados en:", salida)

Archivos guardados en: /content/drive/MyDrive/TRABAJO2 SI/ml_segmentacion
